# Interactive comparison of saved OFO results

This notebook scans the project `results/` tree for run directories containing `records.pkl`. Select **two or more** unique run identifiers, a physical quantity, and the channels to compare.

The curated catalogue includes transmission- and distribution-system voltages, TSO/DSO DER reactive-power infeed, generator reactive power, TSO shunt states, TSO/DSO OLTC positions, AVR setpoints, interface reactive power, tie-line flow, loading, and losses. Every numeric record field is also exposed under **Record field: ...**, so newly added result quantities remain selectable.

> `records.pkl` is loaded with Python pickle. Only use result files produced by this trusted project.

In [1]:
from pathlib import Path
import sys

def locate_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'analysis').is_dir() and (candidate / 'results').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the project root containing analysis/ and results/.')

PROJECT_ROOT = locate_project_root()
RESULTS_ROOT = PROJECT_ROOT / 'results'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'Results root: {RESULTS_ROOT}')

Project root: \\130.83.232.108\homefolders$\mschwenke\Python_Projekte\qOFO_GH
Results root: \\130.83.232.108\homefolders$\mschwenke\Python_Projekte\qOFO_GH\results


In [2]:
import importlib.util
from IPython.display import Markdown, display

if importlib.util.find_spec('ipywidgets') is None:
    display(Markdown(
        '**One-time dependency required:** run `%pip install ipywidgets` in this kernel, '
        'restart the kernel, and rerun the notebook.'
    ))
else:
    print('ipywidgets is available.')

ipywidgets is available.


In [3]:
%matplotlib inline
from analysis.result_comparison import create_dashboard

try:
    dashboard = create_dashboard(RESULTS_ROOT)
except ImportError as exc:
    display(Markdown(f'**Dashboard unavailable:** {exc}'))
else:
    display(dashboard)

## Interpretation notes

- **Small multiples** uses one panel per selected channel and overlays the selected runs. This is normally clearest for controller comparisons.
- **Overlay** puts all selected channels and runs on one axis.
- **Difference to first selected run** interpolates each trace onto the union of timestamps and subtracts the first selected run over their common time interval. Put the intended reference case first.
- Array entries are labelled `item 0`, `item 1`, ... because the current result records store values in controller ordering but do not persist the corresponding physical element indices or names.
- The summary table reports minimum, time-sample mean, maximum, and final value for every plotted run/channel trace.